# Cats Vs. Dogs Classification

Project Sequence:


1.   <small>Data Collection
2.   Data Preprocessing
3.   Data Augmentation
4.   Model Architecture
5.   Feature Extraction
6.   Model Training
7.   Validation & Hyperparameter tuning
8.   Model Evaluation



In [ ]:
!pip install kaggle kagglehub

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

<br>

Data Collection

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset

In [ ]:
!unzip -o microsoft-catsvsdogs-dataset.zip

<br>

Data Preprocessing and Augmentation

<small>Removing corrupted files</small>

In [ ]:
from PIL import Image
import os

dataset_path = "PetImages"

for folder in ["Cat", "Dog"]:
    folder_path = os.path.join(dataset_path, folder)

    for filename in os.listdir(folder_path):
        path = os.path.join(folder_path, filename)

        try:
            img = Image.open(path)
            img.verify()      # Verify it's a valid image
        except Exception:
            print("Removing:", path)
            os.remove(path)

<small>Loading Images</small>

In [ ]:
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image

images = glob("PetImages/Cat/*.jpg")
pictures = glob("PetImages/Dog/*.jpg")

for path in images[:5]:
  img = Image.open(path)

  plt.figure(figsize=(5,5))
  plt.imshow(img)
  plt.axis("off")
  plt.show()

In [ ]:
for paths in pictures[:5]:
  pic = Image.open(paths)

  plt.figure(figsize=(5,5))
  plt.imshow(pic)
  plt.axis("off")
  plt.show()

<br>

<small>Transforms for datasets</small>

In [ ]:
from torchvision import transforms

# standard for MobileNet model
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

<small>Label assignment - ImageFolder</small>

In [ ]:
# ImageFolder

from torchvision.datasets import ImageFolder

dataset = ImageFolder(root="PetImages", transform=None)

train_dataset = ImageFolder(root="PetImages", transform=train_transform)
eval_dataset = ImageFolder(root="PetImages", transform=test_transform)

<small>Train-Validate-Test split</small>

In [ ]:
from torch.utils.data import random_split
import torch

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42) #The Hitchhiker's Guide to the Galaxy

train_dataset, val_dataset, test_dataset = random_split(train_dataset, [train_size, val_size, test_size], generator = generator)

_, val_dataset, test_dataset = random_split(eval_dataset, [train_size, val_size, test_size], generator = generator)

<small>Printing images after applying transform</small>

In [ ]:
import matplotlib.pyplot as plt

for i in range(5):
  pic, label = train_dataset[i]
  print(f"Tensor shape: {pic.shape}")
  plt.figure(figsize=(5,5))
  plt.imshow(pic.permute(1,2,0)) # [C, H, W] - Channels, Height, Width
  print(f"Shape: {pic.shape}")
  plt.axis("off")
  plt.show()

<br>

Model Architecture and Feature Extraction

<small>Load Pretrained model</small>

In [ ]:
from torchvision import models
model = models.mobilenet_v3_small(weights="DEFAULT")

<small>Freeze feature extraction layer</small>

In [ ]:
for param in model.parameters():
  param.requires_grad = False

<small>Replace classifier</small>

In [ ]:
import torch.nn as nn
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 2) #classifying last layer to two classes

<small>Unfreeze selected layers</small>

In [ ]:
import torch

for param in model.features[-5:].parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)

<br>

Model Training

<small>Loss function and Optimizer</small>

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

<small>Epochs and Training</small>

In [ ]:
from torch.utils.data import DataLoader

num_epochs = 20

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
for epoch in range(num_epochs):
  model.train()

  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{num_epochs}, Loss: {loss.item()}")

<br>

Validation & Hyperparameter Tuning

In [ ]:
val_loss = 0
correct = 0
total = 0

from torch.utils.data import DataLoader

eval_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
model.eval()

with torch.no_grad():
  for images, labels in eval_loader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    val_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

val_loss /= len(eval_loader)
val_accuracy = 100 * correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.2f}%")

<br>

Model Evaluation

In [ ]:
test_loss = 0
correct = 0
total = 0

from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model.eval()

with torch.no_grad():
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    val_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

test_loss /= len(test_loader)
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")